# 🚗 Used Cars Price Prediction — Full ML Pipeline
Pipeline مكتمل من Data Cleaning → Preprocessing → Training → Evaluation  
يشمل: LinearRegression, XGBRegressor, LGBMRegressor


## 0️⃣ Imports

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

from sklearn.pipeline        import Pipeline
from sklearn.compose         import ColumnTransformer
from sklearn.impute          import KNNImputer, SimpleImputer
from sklearn.preprocessing   import RobustScaler, OneHotEncoder
from sklearn.linear_model    import LinearRegression
from sklearn.preprocessing   import PolynomialFeatures
from sklearn.model_selection import train_test_split
from sklearn.metrics         import r2_score, mean_absolute_error, mean_squared_error

from category_encoders import TargetEncoder, BinaryEncoder
from xgboost           import XGBRegressor
from lightgbm          import LGBMRegressor, early_stopping

print("✅ All libraries imported successfully")

✅ All libraries imported successfully


## 1️⃣ Data Loading

In [2]:
df_raw = pd.read_csv("vehicles.csv")
print(f"Raw shape: {df_raw.shape}")
df_raw.head(3)

Raw shape: (426880, 26)


,id,url,region,region_url,price,year,manufacturer,model,condition,cylinders,...,size,type,paint_color,image_url,description,county,state,lat,long,posting_date
0,7222695916,https://prescott.craigslist.org/cto/d/prescott...,prescott,https://prescott.craigslist.org,6000,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,az,NaN,NaN,NaN
1,7218891961,https://fayar.craigslist.org/ctd/d/bentonville...,fayetteville,https://fayar.craigslist.org,11900,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,ar,NaN,NaN,NaN
2,7221797935,https://keys.craigslist.org/cto/d/summerland-k...,florida keys,https://keys.craigslist.org,21000,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,fl,NaN,NaN,NaN


## 2️⃣ Data Cleaning
كل خطوات التنظيف في دالة واحدة `clean_data()`

In [3]:
def clean_data(df: pd.DataFrame) -> pd.DataFrame:
    """Full cleaning pipeline: drop → dropna → dedup → filter → fix types."""
    
    # ── 2.1 Drop irrelevant columns ──────────────────────────────────────
    cols_to_drop = [
        'id', 'url', 'region_url', 'VIN', 'image_url',
        'description', 'county', 'lat', 'long',
        'posting_date', 'region', 'size', 'paint_color'
    ]
    df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])

    # ── 2.2 Drop rows with NaN in critical columns ───────────────────────
    critical = ['year', 'odometer', 'manufacturer', 'fuel',
                'transmission', 'title_status', 'model']
    df = df.dropna(subset=critical)

    # ── 2.3 Remove duplicates ────────────────────────────────────────────
    df = df.drop_duplicates()

    # ── 2.4 Remove "other" noise values ─────────────────────────────────
    df = df[
        (df["cylinders"]    != "other") &
        (df["fuel"]         != "other") &
        (df["transmission"] != "other")
    ]

    # ── 2.5 Filter outliers ──────────────────────────────────────────────
    df = df[(df['price']    >= 1_500)   & (df['price']    <= 1_000_000)]
    df = df[(df['year']     >= 2000)    & (df['year']     <= 2022)]
    df = df[(df['odometer'] >= 1_000)   & (df['odometer'] <= 2_000_000)]

    # ── 2.6 Fix cylinders dtype ──────────────────────────────────────────
    df["cylinders"] = (
        df["cylinders"]
        .str.replace("cylinders", "", regex=False)
        .str.strip()
        .astype(float)
    )

    df = df.reset_index(drop=True)
    return df


df = clean_data(df_raw)
print(f"Clean shape: {df.shape}")
df.head(3)

Clean shape: (187921, 13)


,price,year,manufacturer,model,condition,cylinders,fuel,odometer,title_status,transmission,drive,type,state
0,15000,2013.0,ford,f-150 xlt,excellent,6.0,gas,128000.0,clean,automatic,rwd,truck,al
1,35000,2019.0,toyota,tacoma,excellent,6.0,gas,43000.0,clean,automatic,4wd,truck,al
2,19900,2004.0,ford,f250 super duty,good,8.0,diesel,88000.0,clean,automatic,4wd,pickup,al


## 3️⃣ Feature / Target Split & Train-Test Split

In [4]:
X = df.drop("price", axis=1)
y = np.log(df["price"])         

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, shuffle=True
)

X_train = X_train.reset_index(drop=True)
X_test  = X_test.reset_index(drop=True)
y_train = y_train.reset_index(drop=True)
y_test  = y_test.reset_index(drop=True)

print(f"Train: {X_train.shape}  |  Test: {X_test.shape}")

Train: (150336, 12)  |  Test: (37585, 12)


## 4️⃣ Preprocessing Pipeline (ColumnTransformer)

| Column Group | Steps |
|---|---|
| `cylinders` | KNNImputer |
| `odometer`, `year` | KNNImputer → RobustScaler |
| `model` | TargetEncoder |
| `manufacturer`, `state`, `type` | BinaryEncoder |
| `condition`, `drive` | SimpleImputer(most_frequent) → OHE |
| `fuel`, `title_status`, `transmission` | OHE |


In [5]:
# ── Column groups ────────────────────────────────────────────────────────
col_cylinders   = ["cylinders"]
col_robust      = ["odometer", "year"]
col_target_enc  = ["model"]
col_binary_enc  = ["manufacturer", "state", "type"]
col_impute_ohe  = ["condition", "drive"]        
col_ohe_only    = ["fuel", "title_status", "transmission"] 

# ── Sub-pipelines ─────────────────────────────────────────────────────────
pipe_cylinders = Pipeline([
    ("imputer", KNNImputer())
])

pipe_robust = Pipeline([
    ("imputer", KNNImputer()),
    ("scaler",  RobustScaler())
])

pipe_impute_ohe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("ohe",     OneHotEncoder(drop="first", handle_unknown="ignore", sparse_output=False))
])

pipe_ohe = Pipeline([
    ("ohe", OneHotEncoder(drop="first", handle_unknown="ignore", sparse_output=False))
])

# ── ColumnTransformer ─────────────────────────────────────────────────────
preprocessor = ColumnTransformer(
    transformers=[
        ("cylinders",   pipe_cylinders,  col_cylinders),
        ("robust",      pipe_robust,     col_robust),
        ("target_enc",  TargetEncoder(), col_target_enc),
        ("binary_enc",  BinaryEncoder(), col_binary_enc),
        ("impute_ohe",  pipe_impute_ohe, col_impute_ohe),
        ("ohe",         pipe_ohe,        col_ohe_only),
    ],
    remainder="drop",
    verbose_feature_names_out=False
)

print("✅ Preprocessor built")

✅ Preprocessor built


## 5️⃣ Evaluation Helper

In [6]:
def evaluate(name: str, model, X_tr, X_te, y_tr, y_te) -> dict:
    """Fit model and print train/test R2, MAE, RMSE."""
    model.fit(X_tr, y_tr)
    
    y_tr_pred = model.predict(X_tr)
    y_te_pred = model.predict(X_te)
    
    results = {
        "Model":       name,
        "Train R²":    round(r2_score(y_tr, y_tr_pred), 4),
        "Test R²":     round(r2_score(y_te, y_te_pred), 4),
        "Test MAE":    round(mean_absolute_error(y_te, y_te_pred), 4),
        "Test RMSE": round(np.sqrt(mean_squared_error(y_te, y_te_pred)), 4),
    }
    
    print(f"\n{'='*50}")
    print(f"  {name}")
    print(f"{'='*50}")
    for k, v in results.items():
        if k != "Model":
            print(f"  {k:12s}: {v}")
    return results

all_results = []  
print("✅ Evaluate function ready")

✅ Evaluate function ready


## 6️⃣ Model 1 — Linear Regression

In [7]:
pipe_lr = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LinearRegression())])

res = evaluate("Linear Regression", pipe_lr, X_train, X_test, y_train, y_test)
all_results.append(res)


  Linear Regression
  Train R²    : 0.779
  Test R²     : 0.7761
  Test MAE    : 0.2797
  Test RMSE   : 0.3861


## 7️⃣ Model 2 — Polynomial Linear Regression (degree=2)

In [8]:
pipe_poly_lr = Pipeline([
    ("preprocessor", preprocessor),
    ("poly",         PolynomialFeatures(degree=2, include_bias=False)),
    ("model",        LinearRegression())
])

res = evaluate("Poly Linear Regression", pipe_poly_lr, X_train, X_test, y_train, y_test)
all_results.append(res)


  Poly Linear Regression
  Train R²    : 0.8265
  Test R²     : 0.8214
  Test MAE    : 0.2448
  Test RMSE   : 0.3448


## 8️⃣ Model 3 — XGBoost (Default)

In [9]:
pipe_xgb = Pipeline([
    ("preprocessor", preprocessor),
    ("model",        XGBRegressor(objective="reg:squarederror", random_state=42))
])

res = evaluate("XGBoost Default", pipe_xgb, X_train, X_test, y_train, y_test)
all_results.append(res)


  XGBoost Default
  Train R²    : 0.8772
  Test R²     : 0.8533
  Test MAE    : 0.2134
  Test RMSE   : 0.3125


## 9️⃣ Model 4 — Poly_XGBoost (Default)

In [10]:
pipe_poly_xgb =  Pipeline([
    ("preprocessor", preprocessor),
    ("poly",         PolynomialFeatures(degree=2, include_bias=False)),
    ("model",        XGBRegressor(objective="reg:squarederror", random_state=42))
])

res = evaluate("poly XGBoost Default",pipe_poly_xgb, X_train, X_test, y_train, y_test)
all_results.append(res)


  poly XGBoost Default
  Train R²    : 0.8851
  Test R²     : 0.8547
  Test MAE    : 0.2123
  Test RMSE   : 0.3111


## 🔟 Model 5 — XGBoost (Tuned with Early Stopping)

In [11]:
preprocessor_fit = preprocessor 
pipe_xgb_tuned = Pipeline([
    ("preprocessor", preprocessor),
    ("model", XGBRegressor(
        n_estimators       = 1000,
        learning_rate      = 0.05,
        max_depth          = 7,
        subsample          = 0.8,
        colsample_bytree   = 0.8,
        min_child_weight   = 3,
        early_stopping_rounds = 50,
        objective          = "reg:squarederror",
        random_state       = 42
    ))
])

from sklearn.base import clone

prep_clone = clone(preprocessor)
X_train_t  = prep_clone.fit_transform(X_train, y_train)
X_test_t   = prep_clone.transform(X_test)

xgb_tuned = XGBRegressor(
    n_estimators          = 1000,
    learning_rate         = 0.05,
    max_depth             = 7,
    subsample             = 0.8,
    colsample_bytree      = 0.8,
    min_child_weight      = 3,
    early_stopping_rounds = 50,
    objective             = "reg:squarederror",
    random_state          = 42
)

xgb_tuned.fit(
    X_train_t, y_train,
    eval_set=[(X_test_t, y_test)],
    verbose=100
)

y_tr_pred = xgb_tuned.predict(X_train_t)
y_te_pred = xgb_tuned.predict(X_test_t)

res = {
    "Model":     "XGBoost Tuned (Early Stop)",
    "Train R²":  round(r2_score(y_train, y_tr_pred), 4),
    "Test R²":   round(r2_score(y_test,  y_te_pred), 4),
    "Test MAE":  round(mean_absolute_error(y_test, y_te_pred), 4),
    "Test RMSE": round(np.sqrt(mean_squared_error(y_test, y_te_pred)), 4),

}
all_results.append(res)
print("\nXGBoost Tuned → Train R²:", res["Train R²"], " | Test R²:", res["Test R²"])

[0]	validation_0-rmse:0.78595
[100]	validation_0-rmse:0.33871
[200]	validation_0-rmse:0.32434
[300]	validation_0-rmse:0.31441
[400]	validation_0-rmse:0.30803
[500]	validation_0-rmse:0.30398
[600]	validation_0-rmse:0.30124
[700]	validation_0-rmse:0.29908
[800]	validation_0-rmse:0.29737
[900]	validation_0-rmse:0.29572
[999]	validation_0-rmse:0.29472

XGBoost Tuned → Train R²: 0.9074  | Test R²: 0.8695


## 1️⃣1️⃣ Model 6 —> Poly XGBoost (Tuned with Early Stopping)

In [12]:
preprocessor_fit = preprocessor 
pipe_xgb_tuned = Pipeline([
    ("preprocessor", preprocessor),
    ("model", XGBRegressor(
        n_estimators       = 1000,
        learning_rate      = 0.05,
        max_depth          = 7,
        subsample          = 0.8,
        colsample_bytree   = 0.8,
        min_child_weight   = 3,
        early_stopping_rounds = 50,
        objective          = "reg:squarederror",
        random_state       = 42
    ))
])

from sklearn.base import clone

prep_clone = clone(preprocessor)
X_train_t  = prep_clone.fit_transform(X_train, y_train)
X_test_t   = prep_clone.transform(X_test)

poly_xgb_tuned = XGBRegressor(
    n_estimators          = 1000,
    learning_rate         = 0.05,
    max_depth             = 7,
    subsample             = 0.8,
    colsample_bytree      = 0.8,
    min_child_weight      = 3,
    early_stopping_rounds = 50,
    objective             = "reg:squarederror",
    random_state          = 42
)


poly = PolynomialFeatures(degree=2, include_bias=False)

X_train_poly = poly.fit_transform(X_train_t)
X_test_poly  = poly.transform(X_test_t)


poly_xgb_tuned.fit(
    X_train_poly, y_train,
    eval_set=[(X_test_poly, y_test)],
    verbose=100
)

y_tr_pred = poly_xgb_tuned.predict(X_train_poly)
y_te_pred = poly_xgb_tuned.predict(X_test_poly)

res = {
    "Model":     "poly XGBoost Tuned (Early Stop)",
    "Train R²":  round(r2_score(y_train, y_tr_pred), 4),
    "Test R²":   round(r2_score(y_test,  y_te_pred), 4),
    "Test MAE":  round(mean_absolute_error(y_test, y_te_pred), 4),
    "Test RMSE": round(np.sqrt(mean_squared_error(y_test, y_te_pred)), 4),

}
all_results.append(res)
print("\nXGBoost Tuned → Train R²:", res["Train R²"], " | Test R²:", res["Test R²"])

[0]	validation_0-rmse:0.78539
[100]	validation_0-rmse:0.33392
[200]	validation_0-rmse:0.31839
[300]	validation_0-rmse:0.31074
[400]	validation_0-rmse:0.30605
[500]	validation_0-rmse:0.30290
[600]	validation_0-rmse:0.30070
[700]	validation_0-rmse:0.29888
[800]	validation_0-rmse:0.29735
[900]	validation_0-rmse:0.29612
[999]	validation_0-rmse:0.29512

XGBoost Tuned → Train R²: 0.918  | Test R²: 0.8692


## 1️⃣2️⃣ Model 7 — LightGBM (Early Stopping)


In [13]:
lgb_model = LGBMRegressor(n_estimators=1000, learning_rate=0.05, random_state=42)

lgb_model.fit(
    X_train_t, y_train,
    eval_set=[(X_test_t, y_test)],
    callbacks=[early_stopping(50)]
)

y_tr_pred = lgb_model.predict(X_train_t)
y_te_pred = lgb_model.predict(X_test_t)

res = {
    "Model":     "LightGBM (Early Stop)",
    "Train R²":  round(r2_score(y_train, y_tr_pred), 4),
    "Test R²":   round(r2_score(y_test,  y_te_pred), 4),
    "Test MAE":  round(mean_absolute_error(y_test, y_te_pred), 4),
    "Test RMSE": round(np.sqrt(mean_squared_error(y_test, y_te_pred)), 4),

}
all_results.append(res)
print("\nLightGBM → Train R²:", res["Train R²"], " | Test R²:", res["Test R²"])

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.011948 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 606
[LightGBM] [Info] Number of data points in the train set: 150336, number of used features: 36
[LightGBM] [Info] Start training from score 9.411004
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[1000]	valid_0's l2: 0.0938869

LightGBM → Train R²: 0.8835  | Test R²: 0.859


## 1️⃣3️⃣ Model 8 — poly LightGBM (Early Stopping)


In [16]:
poly_lightGBM = LGBMRegressor(n_estimators=1000, learning_rate=0.05, random_state=42)


 
poly_lightGBM.fit(
    X_train_poly, y_train,
    eval_set=[(X_test_poly, y_test)],
    callbacks=[early_stopping(50)]
)

y_tr_pred = poly_lightGBM.predict(X_train_poly)
y_te_pred = poly_lightGBM.predict(X_test_poly)

res = {
    "Model":     "poly LightGBM (Early Stop)",
    "Train R²":  round(r2_score(y_train, y_tr_pred), 4),
    "Test R²":   round(r2_score(y_test,  y_te_pred), 4),
    "Test MAE":  round(mean_absolute_error(y_test, y_te_pred), 4),
    "Test RMSE": round(np.sqrt(mean_squared_error(y_test, y_te_pred)), 4),

}
all_results.append(res)
print("\nLightGBM → Train R²:", res["Train R²"], " | Test R²:", res["Test R²"])

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.143940 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 18622
[LightGBM] [Info] Number of data points in the train set: 150336, number of used features: 615
[LightGBM] [Info] Start training from score 9.411004
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[999]	valid_0's l2: 0.0906374

LightGBM → Train R²: 0.8939  | Test R²: 0.8639


## 📊 Model Comparison — Final Results

In [18]:
results_df = pd.DataFrame(all_results).set_index("Model").sort_values("Test R²", ascending=False)
print("\n", results_df.to_string())
print ("-"*100)
results_df = pd.DataFrame(all_results).set_index("Model").sort_values("Train R²", ascending=False)
print("\n", results_df.to_string())



                                  Train R²  Test R²  Test MAE  Test RMSE
Model                                                                  
XGBoost Tuned (Early Stop)         0.9074   0.8695    0.1970     0.2947
poly XGBoost Tuned (Early Stop)    0.9180   0.8692    0.1973     0.2951
poly LightGBM (Early Stop)         0.8939   0.8639    0.2042     0.3011
LightGBM (Early Stop)              0.8835   0.8590    0.2081     0.3064
poly XGBoost Default               0.8851   0.8547    0.2123     0.3111
XGBoost Default                    0.8772   0.8533    0.2134     0.3125
Poly Linear Regression             0.8265   0.8214    0.2448     0.3448
Linear Regression                  0.7790   0.7761    0.2797     0.3861
----------------------------------------------------------------------------------------------------

                                  Train R²  Test R²  Test MAE  Test RMSE
Model                                                                  
poly XGBoost Tuned (Early Stop)

In [29]:
results_df.index

Index(['poly XGBoost Tuned (Early Stop)', 'XGBoost Tuned (Early Stop)',
       'poly LightGBM (Early Stop)', 'poly XGBoost Default',
       'LightGBM (Early Stop)', 'XGBoost Default', 'Poly Linear Regression',
       'Linear Regression'],
      dtype='object', name='Model')

## 💾 Save Best Model Pipeline

In [30]:
import joblib

joblib.dump(prep_clone,  "preprocessor.joblib")

joblib.dump(pipe_lr,     "linear regression.joblib")
joblib.dump(pipe_poly_lr,"poly linear regression.joblib")


joblib.dump(pipe_xgb,   "XGBoost Deafult.joblib")
joblib.dump(pipe_poly_xgb,   "Poly XGBoost Deafult.joblib")



joblib.dump(xgb_tuned,   "XGBoost optmizing.joblib")
joblib.dump(poly_xgb_tuned,   "Poly XGBoost optmizing.joblib")


joblib.dump(lgb_model,   "LightGBM Model.joblib")
joblib.dump(poly_lightGBM,        "Poly LightGBM Model.joblib")

print("✅ Models saved successfully!")

✅ Models saved successfully!


## 🔮 Inference — How to predict on new data

In [20]:
# ── Load ─────────────────────────────────────────────────────────────────
prep   = joblib.load("preprocessor.joblib")
model  = joblib.load("xgb_tuned_model.joblib")

# ── Example single record ─────────────────────────────────────────────────
new_car = pd.DataFrame([{
    "year":         2018,
    "manufacturer": "ford",
    "model":        "f-150",
    "condition":    "good",
    "cylinders":    8.0,  
    "fuel":         "gas",
    "odometer":     75000,
    "title_status": "clean",
    "transmission": "automatic",
    "drive":        "4wd",
    "type":         "pickup",
    "state":        "ca"
}])

new_car_t       = prep.transform(new_car)
log_price_pred  = model.predict(new_car_t)[0]
price_pred      = np.exp(log_price_pred)

print(f"Predicted Price: ${price_pred:,.0f}")

Predicted Price: $36,095


In [31]:
import pipreqs
! pipreqs

INFO: Not scanning for jupyter notebooks.
Please, verify manually the final list of requirements.txt to avoid possible dependency confusions.
Please, verify manually the final list of requirements.txt to avoid possible dependency confusions.
Please, verify manually the final list of requirements.txt to avoid possible dependency confusions.
INFO: Successfully saved requirements file in E:\final_project\requirements.txt
